In [ ]:
import pandas as pd
import sqlite3
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
##Import the modules
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_squared_error
from sklearn import preprocessing
import matplotlib.pyplot as plt
#-----------------------------------------------------##
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.kernel_ridge import KernelRidge
from sklearn import ensemble
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C
from sklearn.gaussian_process.kernels import DotProduct, WhiteKernel
from sklearn.gaussian_process.kernels import DotProduct, WhiteKernel, Matern

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_squared_error
from sklearn import preprocessing
import matplotlib.pyplot as plt
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
import seaborn as sns
from scipy.stats import norm
from scipy.stats import skew,kurtosis

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
Descriptor = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/GaBP2/GaBP2_2024/5_regression/1_SISSO_features_Complexity_5/ABX2_complexity5_3D.csv")
Descriptor.head(2)

,compound,BandGap,Volume,vand_rad_Avg,vand_rad_Min,vand_rad_Max,cov_rad_Avg,cov_rad_Min,cov_rad_Max,atom_rad_Avg,atom_rad_Min,atom_rad_Max,density_Avg,density_Min,density_Max,d001,d002,d003,bandgap_SISSO
0,ZnGeN2,2.44,181.39,168.33,139,211,104.33,71,122,108.33,65,135,4.42,0.81,7.13,26.743211,-3767.725011,-0.017962,2.735238
1,ZnGeN2,3.02,181.83,168.33,139,211,104.33,71,122,108.33,65,135,4.42,0.81,7.13,26.693666,-3767.725011,-0.017962,2.727145


In [ ]:
Descriptor.keys()

Index(['compound', 'BandGap', 'Volume', 'vand_rad_Avg', 'vand_rad_Min',
       'vand_rad_Max', 'cov_rad_Avg', 'cov_rad_Min', 'cov_rad_Max',
       'atom_rad_Avg', 'atom_rad_Min', 'atom_rad_Max', 'density_Avg',
       'density_Min', 'density_Max', 'd001', 'd002', 'd003', 'bandgap_SISSO'],
      dtype='object')

In [ ]:
ML_data = Descriptor[['compound', 'BandGap','d001', 'd002', 'd003']]
#ML_data = ML_data.round(decimals = 2)
ML_data.head(2)

,compound,BandGap,d001,d002,d003
0,ZnGeN2,2.44,26.743211,-3767.725011,-0.017962
1,ZnGeN2,3.02,26.693666,-3767.725011,-0.017962


## Reviewr Comment explanation

In [ ]:
ML_data_Plot = ML_data.copy()
ML_data_Plot = ML_data_Plot.round(decimals=2)
ML_data_Plot.head(2)

,compound,BandGap,d001,d002,d003
0,ZnGeN2,2.44,26.74,-3767.73,-0.02
1,ZnGeN2,3.02,26.69,-3767.73,-0.02


In [ ]:
ML_data_Plot.keys()

Index(['compound', 'BandGap', 'd001', 'd002', 'd003'], dtype='object')

# ML Model


In [ ]:
ML_data1 = ML_data.drop(['compound'], axis=1)

In [ ]:
ML_Balanced_Data =ML_data1.copy()
ML_Balanced_Data.head(2)

,BandGap,d001,d002,d003
0,2.44,26.743211,-3767.725011,-0.017962
1,3.02,26.693666,-3767.725011,-0.017962


In [ ]:
X1 = ML_Balanced_Data.drop(['BandGap'],axis=1)
X1.head(3)

,d001,d002,d003
0,26.743211,-3767.725011,-0.017962
1,26.693666,-3767.725011,-0.017962
2,26.724029,-3767.725011,-0.017962


In [ ]:
X1.shape

(92, 3)

In [ ]:
Y1 = ML_Balanced_Data['BandGap']
Y1.head(3)

,BandGap
0,2.44
1,3.02
2,2.73


In [ ]:
#X = preprocessing.scale(X1)
X = X1.copy()

# <mark> Repeated K-Fold, which combines K-Fold + Monte-Carlo </mark>

In [ ]:
import numpy as np
import pandas as pd
import csv
import os
from sklearn.model_selection import KFold
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error, r2_score
from shutil import copyfile
from sklearn.model_selection import RepeatedKFold
kf = RepeatedKFold(n_splits=4, n_repeats=25, random_state=42)

fold = 0

for train_index, test_index in kf.split(X):

    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    Y_train, Y_test = Y1.iloc[train_index], Y1.iloc[test_index]

    model = Lasso(alpha=0.00001, max_iter=100000, random_state=fold)

    model.fit(X_train, Y_train)

    # Training prediction
    y_train_predict = model.predict(X_train)
    rmse_train = np.sqrt(mean_squared_error(Y_train, y_train_predict))
    r2_train = r2_score(Y_train, y_train_predict)

    # Test prediction
    y_test_predict = model.predict(X_test)
    rmse_test = np.sqrt(mean_squared_error(Y_test, y_test_predict))
    r2_test = r2_score(Y_test, y_test_predict)

    with open('out.csv', mode='a') as out_file:
        out_writer = csv.writer(out_file)
        out_writer.writerow([fold, rmse_train, r2_train, rmse_test, r2_test])

    fold += 1


copyfile('out.csv', 'cross-validation-out.csv')

data_stat = pd.read_csv('out.csv', names=["Fold","RMSE-Train","R2-Train","RMSE-Test","R2-Test"])

print(data_stat.describe())

try:
    os.remove("out.csv")
except OSError:
    pass

             Fold  RMSE-Train    R2-Train   RMSE-Test     R2-Test
count  100.000000  100.000000  100.000000  100.000000  100.000000
mean    49.500000    0.309507    0.927818    0.315483    0.914795
std     29.011492    0.020462    0.011433    0.059854    0.039869
min      0.000000    0.251328    0.902834    0.163383    0.784426
25%     24.750000    0.298502    0.922396    0.274064    0.888670
50%     49.500000    0.312459    0.927661    0.314204    0.921021
75%     74.250000    0.324882    0.935249    0.354454    0.939909
max     99.000000    0.348213    0.954346    0.449808    0.979901


In [ ]:
#              Fold  RMSE-Train    R2-Train   RMSE-Test     R2-Test
# count  100.000000  100.000000  100.000000  100.000000  100.000000
# mean    49.500000    0.309507    0.927818    0.315483    0.914795
# std     29.011492    0.020462    0.011433    0.059854    0.039869
# min      0.000000    0.251328    0.902834    0.163383    0.784426
# 25%     24.750000    0.298502    0.922396    0.274064    0.888670
# 50%     49.500000    0.312459    0.927661    0.314204    0.921021
# 75%     74.250000    0.324882    0.935249    0.354454    0.939909
# max     99.000000    0.348213    0.954346    0.449808    0.979901

In [ ]:
X_train.shape

(69, 3)

In [ ]:
X_test.shape

(23, 3)

In [ ]:
23/(69+23)

0.25